In [ ]:
# [COLAB_BOOTSTRAP_v2_modular]
# Mount Drive + clone upstream moment_detr + extract features to LOCAL SSD.
# Run once at the top of any modular notebook. Idempotent within a session.

import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

# --- 1. Drive --------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/thumbnail_project')
REPO_DIR   = DRIVE_ROOT / 'thumbnail-consistency'
assert REPO_DIR.exists(), (
    f"Upload your repo folder to {REPO_DIR} first. "
    f"Found nothing at that path."
)

# --- 2. Put repo on sys.path (so `from src.*` works) -----------------------
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- 2b. Symlink data dirs INTO the repo so relative paths work ------------
# Notebooks reference 'processed/X.jsonl', 'outputs/Y.png' etc., assuming
# those dirs live INSIDE the repo. In your Drive layout they sit ALONGSIDE
# the repo. Symlinks bridge that gap.
DATA_DIRS_TO_LINK = [
    'processed', 'outputs',
    'qvh_videos', 'qvh_frames', 'qvh_thumbnails',
    'qvh_frames_within_neg', 'qvh_frames_multi',
    'checkpoints',
]
for d in DATA_DIRS_TO_LINK:
    src = DRIVE_ROOT / d
    dst = REPO_DIR / d
    if not src.exists():
        continue
    if dst.exists() or dst.is_symlink():
        continue
    try:
        os.symlink(src, dst)
        print(f"  symlinked  {d}/  ->  {src}")
    except OSError as e:
        print(f"  WARN: couldn't symlink {d}: {e}")

# --- 3. Extract M1 features to LOCAL SSD (NOT to Drive) --------------------
# Drive cannot reliably store 19k small .npz files; we extract every session.
ARCHIVE   = DRIVE_ROOT / 'moment_detr_features.tar.gz'
LOCAL_OUT = Path('/content/features_local')


def _find_feature_dirs(root):
    clip = text = None
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            p = Path(r) / d
            if d == 'clip_features' and len(list(p.glob('*.npz'))) > 1000:
                clip = p
            if d == 'clip_text_features' and len(list(p.glob('*.npz'))) > 1000:
                text = p
    return clip, text


CLIP_FEAT_DIR, TEXT_FEAT_DIR = (None, None)
if LOCAL_OUT.exists():
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(LOCAL_OUT)

if CLIP_FEAT_DIR and TEXT_FEAT_DIR:
    print(f"[features] already extracted this session.")
else:
    assert ARCHIVE.exists(), f"Archive missing: {ARCHIVE}"
    if LOCAL_OUT.exists():
        shutil.rmtree(LOCAL_OUT)
    LOCAL_OUT.mkdir(parents=True)
    print(f"[features] extracting {ARCHIVE.stat().st_size/1e9:.2f} GB -> {LOCAL_OUT}")
    t0 = time.time()
    with tarfile.open(ARCHIVE) as tf:
        tf.extractall(LOCAL_OUT)
    print(f"[features] extracted in {time.time()-t0:.0f}s")
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(LOCAL_OUT)

assert CLIP_FEAT_DIR and TEXT_FEAT_DIR, "Feature dirs not found after extraction."

# --- 4. Clone upstream Moment-DETR (used by src.models.moment_detr) --------
MD_DIR = Path('/content/moment_detr_upstream')
if not MD_DIR.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/jayleicn/moment_detr.git',
                    str(MD_DIR)], check=True)
if str(MD_DIR) not in sys.path:
    sys.path.insert(0, str(MD_DIR))

# --- 5. Expose path variables for downstream cells -------------------------
# Any cell that reads configs/*.yaml can override:
#   cfg['paths']['feature_dir']   = str(CLIP_FEAT_DIR)
#   cfg['paths']['text_feat_dir'] = str(TEXT_FEAT_DIR)
# (configs already point at /content/features_local/... so usually no override needed.)

# --- 6. Verification report -------------------------------------------------
import torch
print()
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (WARN: slow)'}")
print(f"clip_features   : {CLIP_FEAT_DIR}  ({len(list(CLIP_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"clip_text_feats : {TEXT_FEAT_DIR}  ({len(list(TEXT_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"src/ importable : {(REPO_DIR / 'src' / '__init__.py').exists()}")
print()
print("Bootstrap done. You can now run the rest of the notebook.")


# Train ClipCap-M2 (★ the only trained component for V1)

Owner: Person B (M2)
Day 1 deliverable: working model saved to `outputs/checkpoints/clipcap_best.pt`.

Selected by best **CLIP-text similarity** on `m2_val_pairs.jsonl`, NOT BLEU.

In [ ]:
import sys, json, math, time
from pathlib import Path

# In Colab: drive.mount + cd to project root before this cell.
PROJECT = Path('.').resolve()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import torch, numpy as np
from torch.utils.data import DataLoader
from src.utils.io import load_jsonl, load_yaml
from src.utils.seed import set_seed
from src.models.clipcap import ClipCapM2
from src.models.encoders import get_clip, encode_text
from src.data.m2_dataset import M2PairsDataset, collate_m2

cfg = load_yaml('configs/clipcap.yaml')
set_seed(cfg['seed'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [ ]:
# Build model
model = ClipCapM2(**cfg['model']).to(device)
print(f'Trainable params: {model.num_trainable():,}')

# Load shared CLIP for image encoding inside the dataset
clip_model, clip_proc, clip_tok, _ = get_clip()

train_rec = load_jsonl(cfg['data']['train_jsonl'])
val_rec   = load_jsonl(cfg['data']['val_jsonl'])
print(f'train: {len(train_rec)}, val: {len(val_rec)}')

train_ds = M2PairsDataset(train_rec, model.tokenizer, clip_proc, clip_model,
                           device=device,
                           max_caption_tokens=cfg['data']['max_caption_tokens'],
                           cached_emb_path='processed/m2_image_emb.npz')
val_ds   = M2PairsDataset(val_rec,   model.tokenizer, clip_proc, clip_model,
                           device=device,
                           max_caption_tokens=cfg['data']['max_caption_tokens'])

train_dl = DataLoader(train_ds, batch_size=cfg['train']['batch_size'],
                       shuffle=True, collate_fn=collate_m2, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=cfg['train']['batch_size'],
                       shuffle=False, collate_fn=collate_m2, num_workers=0)

In [ ]:
# Optimizer + LR scheduler (linear warmup -> cosine)
optim = torch.optim.AdamW(model.trainable_parameters(),
                           lr=cfg['train']['lr'],
                           weight_decay=cfg['train']['weight_decay'])

total_steps  = len(train_dl) * cfg['train']['epochs']
warmup_steps = max(1, int(total_steps * cfg['train']['warmup_ratio']))
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * p))
sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

loss_fn = torch.nn.CrossEntropyLoss(
    ignore_index=model.tokenizer.pad_token_id,
    label_smoothing=cfg['train']['label_smoothing'],
)

In [ ]:
@torch.no_grad()
def eval_clip_text_sim(model, val_dl, n_max=200):
    """Generate one query per val frame, compare to GT via CLIP-text cosine."""
    model.eval()
    gens, gts = [], []
    for batch in val_dl:
        cf = batch['clip_feat'].to(device)
        outs = model.generate(cf, max_new_tokens=20, num_return_sequences=1,
                               do_sample=False, num_beams=3)
        gens.extend([o[0] for o in outs])
        gts.extend(batch['query'])
        if len(gens) >= n_max:
            break
    if not gens: return 0.0
    e_g = encode_text(gens).numpy()
    e_t = encode_text(gts).numpy()
    cos = (e_g * e_t).sum(axis=1)  # both L2-normalized
    return float(cos.mean())

In [ ]:
ckpt_dir = Path(cfg['output']['ckpt_dir'])
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_path = ckpt_dir / cfg['output']['best_name']

best_sim = -1.0
patience = cfg['train']['early_stop_patience']
since_best = 0
history = []

for epoch in range(cfg['train']['epochs']):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for batch in train_dl:
        cf = batch['clip_feat'].to(device)
        ids = batch['input_ids'].to(device)
        am  = batch['attention_mask'].to(device)
        # Teacher forcing: predict token t+1 from prefix + tokens[0..t].
        logits = model(cf, ids[:, :-1], am[:, :-1])
        loss = loss_fn(logits.reshape(-1, logits.size(-1)),
                        ids[:, 1:].reshape(-1))
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.trainable_parameters(),
                                        cfg['train']['grad_clip'])
        optim.step(); sched.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / max(1, len(train_dl))
    sim = eval_clip_text_sim(model, val_dl)
    dt = time.time() - t0
    history.append({'epoch': epoch, 'train_loss': avg_loss, 'val_clip_sim': sim, 'dt_s': dt})
    print(f'epoch {epoch:02d} | loss {avg_loss:.4f} | val CLIP-sim {sim:.4f} | {dt:.1f}s')

    if sim > best_sim:
        best_sim = sim; since_best = 0
        torch.save({'model': model.state_dict(),
                    'cfg': cfg, 'history': history, 'best_sim': best_sim}, best_path)
        print(f'  -> saved {best_path}')
    else:
        since_best += 1
        if since_best >= patience:
            print(f'Early stop at epoch {epoch} (best CLIP-sim={best_sim:.4f})')
            break

with open(ckpt_dir / 'training_log.json', 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

In [ ]:
# Generate K=3 nucleus-sampled queries for every consistency_eval pair.
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()

consist = load_jsonl('processed/consistency_eval.jsonl')
# Use the THUMB's frame_path as input (positives) or the negative's frame (negs).
from src.models.encoders import encode_image
out_path = Path('outputs/predictions/m2_generated.jsonl')
out_path.parent.mkdir(parents=True, exist_ok=True)

K = cfg['generation']['num_return_sequences']
results = []
with open(out_path, 'w', encoding='utf-8') as f:
    BATCH = 16
    for i in range(0, len(consist), BATCH):
        rows = consist[i:i+BATCH]
        emb, ok = encode_image([r['frame_path'] for r in rows])
        if emb.shape[0] == 0: continue
        outs = model.generate(emb.to(device),
                               max_new_tokens=cfg['generation']['max_new_tokens'],
                               num_return_sequences=K,
                               do_sample=True,
                               top_p=cfg['generation']['top_p'],
                               temperature=cfg['generation']['temperature'],
                               no_repeat_ngram_size=cfg['generation']['no_repeat_ngram_size'])
        for r, qs in zip(rows, outs):
            rec = {'pair_id': r['pair_id'], 'queries': qs}
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            results.append(rec)

print(f'Wrote {len(results)} generations -> {out_path}')